# Sliding-Window Choice Probability

Compute per-session P(right choice) in a sliding window. Right = 1, left = 0, no-response trials excluded. Same windowing semantics as `compute_sliding_reward_rate`.

In [ ]:
# Environment setup
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

MODULE_PATH = Path('/root/capsule/src/aind_dft_ephys_analysis')
if str(MODULE_PATH) not in sys.path:
    sys.path.insert(0, str(MODULE_PATH))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from nwb_utils import NWBUtils
from behavior_utils import (
    compute_sliding_choice_probability,
    compute_sliding_choice_probability_from_nwb,
)
print('Loaded modules.')

In [ ]:
# Sessions to analyze
sessions = [
    'ecephys_844034_2026-05-05_12-26-26_sorted_2026-05-13_16-51-38',
    'ecephys_844034_2026-05-06_12-31-42_sorted_2026-05-10_00-09-35',
    'ecephys_844034_2026-05-07_12-26-08_sorted_2026-05-10_22-15-43',
    'ecephys_844036_2026-05-04_16-06-43_sorted_2026-05-09_21-00-45',
    'ecephys_844036_2026-05-05_16-08-08_sorted_2026-05-19_17-42-51',
    'ecephys_844036_2026-05-06_16-27-46_sorted_2026-05-10_00-06-13',
]

# Sliding-window parameters
window      = 20          # trials
step        = 1           # one value per trial
causal      = False       # False -> centered; True -> trailing/causal
side        = 'right'     # P(right) ; switch to 'left' for P(left) = 1 - P(right)
min_periods = None        # None -> require full window (no edge values)

In [ ]:
# Load NWBs and compute sliding choice probability per session
results = {}
for sess in sessions:
    try:
        nwb = NWBUtils.read_ophys_or_behavior_nwb(session_name=sess)
        if nwb is None:
            print(f'[skip] no NWB for {sess}')
            continue
        try:
            res = compute_sliding_choice_probability_from_nwb(
                nwb,
                window=window,
                step=step,
                min_periods=min_periods,
                causal=causal,
                side=side,
            )
        finally:
            try:
                nwb.io.close()
            except Exception:
                pass
        results[sess] = res
        finite = np.isfinite(res['choice_prob'])
        mean_cp = float(np.nanmean(res['choice_prob'])) if finite.any() else float('nan')
        print(f'{sess}: n_trials={res["trial_index"].size}, mean_P({side})={mean_cp:.3f}')
    except Exception as e:
        print(f'[error] {sess}: {e}')

In [ ]:
# Per-session plots: choice probability vs trial index and distribution
# Bin edges aligned to the discrete grid k/window so every possible value
# falls in its own bin.
bin_width = 1.0 / window
bins = np.arange(-bin_width / 2, 1.0 + bin_width, bin_width)

for sess, res in results.items():
    fig, axes = plt.subplots(1, 2, figsize=(12, 3.2),
                             gridspec_kw={'width_ratios': [3, 1]})

    ax = axes[0]
    ax.plot(res['trial_index'], res['choice_prob'], color='tab:purple', lw=1.2)
    mu = float(np.nanmean(res['choice_prob']))
    ax.axhline(0.5, color='gray', ls=':', lw=0.8)
    ax.axhline(mu, color='k', ls='--', lw=0.8, label=f'mean={mu:.2f}')
    ax.set_xlabel('Trial index')
    ax.set_ylabel(f'P({side}) (window={window})')
    ax.set_title(f'{sess}')
    ax.set_ylim(0, 1)
    ax.legend(loc='lower right', fontsize=8)

    ax = axes[1]
    vals = res['choice_prob'][np.isfinite(res['choice_prob'])]
    if vals.size:
        ax.hist(vals, bins=bins, orientation='horizontal',
                color='tab:purple', alpha=0.75, edgecolor='k')
        med = float(np.median(vals))
        ax.axhline(med, color='tab:red', ls='--', lw=1.0, label=f'median={med:.2f}')
        ax.axhline(mu,  color='k',       ls='--', lw=0.8, label=f'mean={mu:.2f}')
        ax.axhline(0.5, color='gray',    ls=':',  lw=0.8)
        ax.legend(loc='lower right', fontsize=7)
    ax.set_ylim(0, 1)
    ax.set_xlabel('Count')
    ax.set_title('Distribution')

    plt.tight_layout()
    plt.show()

In [ ]:
# Overlay: all sessions on one axis
fig, ax = plt.subplots(figsize=(9, 4))
for sess, res in results.items():
    ax.plot(res['trial_index'], res['choice_prob'], lw=1.0, alpha=0.8,
            label=sess.split('_sorted')[0])
ax.axhline(0.5, color='gray', ls=':', lw=0.8)
ax.set_xlabel('Trial index')
ax.set_ylabel(f'P({side}) (window={window})')
ax.set_ylim(0, 1)
ax.legend(fontsize=7, loc='lower right')
ax.set_title('Sliding-window choice probability across sessions')
plt.tight_layout(); plt.show()

In [ ]:
# =============================================================================
# Distribution of sliding-window choice probabilities
#   Left panel : per-session histograms overlaid (transparent)
#   Right panel: pooled histogram across all sessions
# =============================================================================
bin_width = 1.0 / window
bins = np.arange(-bin_width / 2, 1.0 + bin_width, bin_width)

fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))

ax = axes[0]
cmap = plt.get_cmap('tab10')
for k, (sess, res) in enumerate(results.items()):
    vals = res['choice_prob']
    vals = vals[np.isfinite(vals)]
    if vals.size == 0:
        continue
    ax.hist(vals, bins=bins, density=True, alpha=0.35,
            color=cmap(k % 10), label=sess.split('_sorted')[0])
    ax.axvline(np.median(vals), color=cmap(k % 10), ls='--', lw=0.8)
ax.axvline(0.5, color='gray', ls=':', lw=0.8)
ax.set_xlabel(f'P({side}) (window={window})')
ax.set_ylabel('Density')
ax.set_title('Per-session distributions (dashed = median)')
ax.set_xlim(0, 1)
ax.legend(fontsize=7, loc='upper left')

ax = axes[1]
pooled = np.concatenate([
    res['choice_prob'][np.isfinite(res['choice_prob'])]
    for res in results.values()
]) if results else np.array([])
if pooled.size:
    ax.hist(pooled, bins=bins, density=True, color='tab:gray',
            edgecolor='k', alpha=0.85)
    med = float(np.median(pooled))
    mu  = float(np.mean(pooled))
    ax.axvline(med, color='tab:red',  ls='--', lw=1.0, label=f'median={med:.2f}')
    ax.axvline(mu,  color='tab:blue', ls='--', lw=1.0, label=f'mean={mu:.2f}')
    ax.axvline(0.5, color='gray',     ls=':',  lw=0.8)
    ax.legend(loc='upper left', fontsize=8)
ax.set_xlabel(f'P({side}) (window={window})')
ax.set_ylabel('Density')
ax.set_title(f'Pooled across {len(results)} session(s) (n={pooled.size} windows)')
ax.set_xlim(0, 1)

plt.tight_layout(); plt.show()

summary = pd.DataFrame([
    {
        'session': sess,
        'n_windows': int(np.isfinite(res['choice_prob']).sum()),
        'mean':   float(np.nanmean(res['choice_prob'])),
        'median': float(np.nanmedian(res['choice_prob'])),
        'std':    float(np.nanstd(res['choice_prob'])),
        'min':    float(np.nanmin(res['choice_prob'])),
        'max':    float(np.nanmax(res['choice_prob'])),
    }
    for sess, res in results.items()
])
summary